# 04 · Panel Construction

Joins DGStats, CAISO monthly, DeepSolar, and utility territory map into `panel_for_regression.parquet`.

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

PROC = ROOT / 'data' / 'processed'
EXT = ROOT / 'data' / 'external'

import pandas as pd
import geopandas as gpd
import numpy as np
from feature_engineering import build_panel

## 4.1 Build ZIP → Utility Territory Map

In [2]:
utility_gdf = gpd.read_file(EXT / 'utility_territories.geojson')
print(utility_gdf.crs, utility_gdf.shape)
utility_gdf.head(2)

EPSG:4326 (3, 48)


,Acronym,Utility,AgencyNum,Type,URL,Phone,Address,HIFLD_ID,Sales_GWh_,Sales_GWh1,...,Sales_G_29,Sales_G_30,Sales_G_31,Sales_G_32,Sales_G_33,Sales_G_34,OnlineName,Audit,utility_name,geometry
0,SCE,Southern California Edison,86250.0,IOU,https://www.sce.com/,(818) 302-1212,"2244 Walnut Grove Avenue Rosemead, CA 91770-3714",17609,71016.547845,69272.28376,...,81054.000005,81132.999996,82971.000004,76840.077169,0.0,0.0,None,None,Southern California Edison,"MULTIPOLYGON Z (((-118.60361 33.47801 0, -118...."
1,PG&E,Pacific Gas & Electric Company,71021.0,IOU,https://www.pge.com/,(415) 973-7000,"77 Beale Street San Francisco, CA 94105",14328,70036.326445,70483.38716,...,78518.835139,78438.000003,77887.000005,72932.833237,0.0,0.0,None,None,Pacific Gas & Electric Company,"MULTIPOLYGON Z (((-120.87195 35.21664 0, -120...."


In [3]:
# Read CA ZCTA shapefile — use Census TIGER ZCTA5
# If not yet downloaded, note the path and skip for now
zcta_path = EXT / 'ca_zcta.geojson'
if zcta_path.exists():
    zcta_gdf = gpd.read_file(zcta_path)
    zcta_gdf = zcta_gdf.to_crs(utility_gdf.crs)
    zcta_centroids = zcta_gdf.copy()
    zcta_centroids['geometry'] = zcta_gdf.geometry.centroid
    joined = gpd.sjoin(zcta_centroids[['ZCTA5CE20', 'geometry']], utility_gdf[['utility_name', 'geometry']], how='left', predicate='within')
    zip_utility = joined[['ZCTA5CE20', 'utility_name']].rename(columns={'ZCTA5CE20': 'zip_code', 'utility_name': 'utility'})
    zip_utility['zip_code'] = zip_utility['zip_code'].astype(str).str.zfill(5)
    zip_utility.to_parquet(PROC / 'zip_utility_map.parquet', index=False)
    print(f'ZIP → utility map: {len(zip_utility):,} ZIPs')
else:
    print(f'ZCTA shapefile not found at {zcta_path}. Using placeholder empty map.')
    zip_utility = pd.DataFrame(columns=['zip_code', 'utility'])

ZIP → utility map: 2,045 ZIPs


/var/folders/6s/xmd0chzj1pqcst4_3lrvmq380000gn/T/ipykernel_23046/2124578364.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zcta_centroids['geometry'] = zcta_gdf.geometry.centroid


## 4.2 Merge into Regression Panel

In [4]:
dgstats_panel = pd.read_parquet(PROC / 'dgstats_panel.parquet')
caiso_monthly = pd.read_parquet(PROC / 'caiso_monthly.parquet')
deepsolar_zip = pd.read_parquet(PROC / 'deepsolar_zip.parquet')

# Rename year column if needed
if 'install_year' in dgstats_panel.columns:
    dgstats_panel = dgstats_panel.rename(columns={'install_year': 'year'})

panel = build_panel(dgstats_panel, caiso_monthly, deepsolar_zip, zip_utility)
print(f'Panel: {len(panel):,} rows | {panel["zip_code"].nunique():,} ZIPs × {panel["year"].nunique()} years × 12 months')
panel.head(3)

Panel: 174,744 rows | 1,618 ZIPs × 9 years × 12 months


,zip_code,year,month,annual_capacity_kw,install_count,storage_paired_count,btm_capacity_kw,demand_gwh,solar_gwh,wind_gwh,...,solar_system_count,solar_panel_area_divided_by_area,utility,btm_lag1,btm_lag2,log_btm_lag1,log_btm_lag1_sq,ramp_magnitude_mwh,curtailment_days_per_month,negative_lmp_hours_per_month
0,90001,2015,1,109.09698,23.0,0.0,109.09698,17803.362,1253.833333,1015.0,...,193.0,17132.708733,Southern California Edison,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,90001,2015,2,109.09698,23.0,0.0,109.09698,15997.661,1253.833333,1015.0,...,193.0,17132.708733,Southern California Edison,NaN,NaN,NaN,NaN,1805701.0,0.0,0.0
2,90001,2015,3,109.09698,23.0,0.0,109.09698,18211.601,1253.833333,1015.0,...,193.0,17132.708733,Southern California Edison,NaN,NaN,NaN,NaN,2213940.0,0.0,0.0


## 4.3 Balance Validation

In [5]:
expected_combos = panel['zip_code'].nunique() * panel['year'].nunique() * 12
actual = len(panel)
print(f'Expected (balanced): {expected_combos:,} | Actual: {actual:,}')
if actual < expected_combos:
    missing = expected_combos - actual
    print(f'Missing {missing:,} rows — check for gaps in DGStats coverage')
else:
    print('Panel is balanced.')

Expected (balanced): 174,744 | Actual: 174,744
Panel is balanced.


In [6]:
panel.to_parquet(PROC / 'panel_for_regression.parquet', index=False)
print('Saved panel_for_regression.parquet')

Saved panel_for_regression.parquet
